In [1]:
"""
Testing the fb_tools functions
"""

# standard imports
import os
import geopandas as gpd
import pandas as pd

from pathlib import Path
from os.path import join
from fb_tools.weather import build_pyrome_wind_cells

# directories
# use the current working directory
projdir = Path.cwd().parents[1] # moves up two, outside code directory
print(f"Project directory set to: {projdir}")
print(os.getcwd())

# environ vars
proj_crs = 26913  # NAD83 UTM Zone 13N

print("Ready to go !")

Project directory set to: /Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper
/Users/mcc/Library/CloudStorage/Box-Box/MCC/fire_modeling/FM_PythonWrapper/code/notebooks
Ready to go !


In [2]:
# --- Load the fire data
fod = gpd.read_file(join(projdir,'data/spatial/raw/fpa_fod/SHP/Fires_ClassDEFG_CO_Pyromes.shp'))
fod['ignition_date'] = pd.to_datetime(fod['DISCOVERY_'])
fod = fod[fod['FIRE_YEAR']>=2016]

# --- Load Pyromes and join to FOD
pyromes = gpd.read_file(join(projdir,'data/spatial/raw/boundaries/CO_Pyromes.gpkg'))
pyromes = pyromes.to_crs(fod.crs)
fod = gpd.sjoin(fod, pyromes[['PYROME','geometry']], how='left', predicate='within')
print(f"Number of FOD: {len(fod)}\nNumber of Pyromes: {len(pyromes)}")

Number of FOD: 957
Number of Pyromes: 9


In [ ]:
# --- Build FSPro wind cells based on historic fire events
wind_cells = build_pyrome_wind_cells(
    fod, # fire locations
    cache_dir='data/weather/hrrr_cache',
    out_dir='data/weather/pyrome_wind',
)

Building wind climatology from HRRR ...
  957 / 957 fires in HRRR period (2016+)
  19140 raw schedule entries → 6352 unique (date, hour) downloads
  [1/6352] 2016-01-14 19:00  (0 obs so far)
  HRRR grid shape: (1059, 1799) (1,905,141 grid points)
  [7/6352] SKIP 2016-01-15 21:00: Processing failed: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
  [13/6352] SKIP 2016-01-17 19:00: 
No index file was found for None
Download the full file first (with `H.download()`).
You will need to remake the Herbie object (H = `Herbie()`)
or delete this cached property: `del H.index_as_dataframe()`
  [17/6352] SKIP 2016-01-18 19:00: 
No index file was found for None
Download the full file first (with `H.download()`).
You will need to remake the Herbie object (H = `Herbie()`)
or delete this cached property: `del H.index_as_dataframe()`


In [ ]:
wind_cells